# Notebook 1: Einführung und Daten

**Statistik für Data Science** · Kurs 3,906 · Bachelor Computer Science · HS2026
Universität St. Gallen

---
### Projektteam:
- Andrin Greitmann
- Noah Appenzeller
- Manuel Schmid

---

Dieses Notebook führt die Inhalte von **VL01 (Einführung & Daten)** in Python aus. 

## 0. Setup
Importiert `pandas`, `numpy`, `seaborn` und `matplotlib` und setzt den Seed.

In [16]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Seed: Startwert für den Zufallsgenerator. Gleicher Seed bei jedem Durchlauf damit Resultate deterministisch & reproduzierbar sind. 
# Den Wert selbst haben wir beliebig gewählt, wir haben ihn als erstes vor der Analyse gewählt und danach nicht mehr geändert.
# Der Seed wird für alles Zufällige in allen Notebooks verwendet.
SEED = 2001
RNG = np.random.default_rng(SEED)

# Plot-Defauls
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (7.5, 4.2)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11

In [17]:
import sys

print("Python  ", sys.version.split()[0])
for modul in (np, pd, sns):
    print(f"{modul.__name__:8s}", modul.__version__)
print("Seed    ", SEED)

Python   3.14.4
numpy    2.5.3
pandas   3.0.6
seaborn  0.13.2
Seed     2001


### Unseren Datensatz laden

In [18]:
from ucimlrepo import fetch_ucirepo

# Datensatz direkt aus dem UCI Repository laden (ID 117 = Census-Income KDD)
census_income_kdd = fetch_ucirepo(id=117)

# Etwas spezielles an unserem Datenset: ucimlrepo liefert die Daten aufgeteilt in Features und Target. 
# (Wahrscheinlich für Machine Learning Projekte/Algorithmen)
# Target ist das Einkommen (über/unter 50K), Features sind alle anderen Merkmale.
X = census_income_kdd.data.features
y = census_income_kdd.data.targets

# Für die EDA wollten wir alle Spalten in einem DataFrame damit das Einkommen mit den anderen Merkmalen verglichen werden kann.
# join() funktioniert hier zum Glück recht einfach, weil X und y dieselben Zeilen in derselben Reihenfolge haben.
census = X.join(y)

# Wir wollen die Spalten umbenennen, sodass es ersichtlicher ist, mit was wir arbeiten
spalten_umbenennung = {
    "AAGE": "AGE", "ACLSWKR": "CLASS_OF_WORKER", "ADTINK": "INDUSTRY_CODE",      
    "ADTOCC": "OCCUPATION_CODE", "AHGA": "EDUCATION", "AHSCOL": "ENROLLED_IN_EDU",
    "AMARITL": "MARITAL_STATUS", "AMJIND": "MAJOR_INDUSTRY", "AMJOCC": "MAJOR_OCCUPATION",
    "ARACE": "RACE", "AREORGN": "HISPANIC_ORIGIN", "ASEX": "SEX",
    "AUNMEM": "LABOR_UNION_MEMBER", "AUNTYPE": "UNEMPLOYMENT_REASON", "AWKSTAT": "EMPLOYMENT_STATUS",
    "CAPGAIN": "CAPITAL_GAINS", "GAPLOSS": "CAPITAL_LOSSES", "DIVVAL": "DIVIDENDS",
    "FILESTAT": "TAX_FILER_STATUS", "GRINREG": "PREV_REGION", "GRINST": "PREV_STATE",
    "HHDFMX": "HOUSEHOLD_STAT_DETAILED", "HHDREL": "HOUSEHOLD_SUMMARY", "MARSUPWRT": "WEIGHT",
    "MIGMTR1": "MIGRATION_MSA", "MIGMTR3": "MIGRATION_REG", "MIGMTR4": "MIGRATION_WITHIN_REG",
    "MIGSAME": "SAME_HOUSE_1YR", "MIGSUN": "MIGRATION_SUNBELT", "NOEMP": "NUM_EMPLOYEES",
    "PARENT": "PARENTS_PRESENT", "PEFNTVTY": "BIRTH_COUNTRY_FATHER", "PEMNTVTY": "BIRTH_COUNTRY_MOTHER",
    "PENATVTY": "BIRTH_COUNTRY_SELF", "PRCITSHP": "CITIZENSHIP", "SEOTR": "SELF_EMPLOYED",
    "VETQVA": "VET_QUESTIONNAIRE", "VETYN": "VETERANS_BENEFITS", "WKSWORK": "WEEKS_WORKED",
    "AHRSPAY": "WAGE_PER_HOUR", "year": "YEAR", "income": "INCOME"
}
census = census.rename(columns=spalten_umbenennung)

# Metadaten: Herkunft, Beschreibung und Erhebung des Datensatzes (Damit wir auf Bias analysieren können)
print("Metadata: " + str(census_income_kdd.metadata))

print(f"Rows: {census.shape[0]}, Columns: {census.shape[1]}")

# Variablenbeschreibung: Name, Typ und Bedeutung jeder Spalte (Grundlage um später Messniveau zu bestimmen)
print("Variable descriptions:")

display(census_income_kdd.variables)

Metadata: {'uci_id': 117, 'name': 'Census-Income (KDD)', 'repository_url': 'https://archive.ics.uci.edu/dataset/117/census+income+kdd', 'data_url': 'https://archive.ics.uci.edu/static/public/117/data.csv', 'abstract': 'This data set contains weighted census data extracted from the 1994 and 1995 current population surveys conducted by the U.S. Census Bureau.', 'area': 'Social Science', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 299285, 'num_features': 41, 'feature_types': ['Categorical', 'Integer'], 'demographics': ['Age', 'Occupation', 'Education Level', 'Marital Status', 'Race', 'Sex', 'Other', 'Nationality', 'Income'], 'target_col': ['income'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 2000, 'last_updated': 'Fri Mar 08 2024', 'dataset_doi': '10.24432/C5N30T', 'creators': [], 'intro_paper': None, 'additional_info': {'summary': "This data set contains weighted census data extracted

,name,role,type,demographic,description,units,missing_values
0,AAGE,Feature,Integer,Age,age,None,no
1,ACLSWKR,Feature,Categorical,NaN,class of worker,None,no
2,ADTINK,Feature,Integer,NaN,industry code,None,no
3,ADTOCC,Feature,Integer,Occupation,occupation code,None,no
4,AHGA,Feature,Integer,Education Level,education,None,no
5,AHSCOL,Feature,Categorical,Education Level,enrolled in edu last week,None,no
6,AMARITL,Feature,Categorical,Marital Status,marital status,None,no
7,AMJIND,Feature,Categorical,NaN,major industry code,None,no
8,AMJOCC,Feature,Categorical,Occupation,major occupation code,None,no
9,ARACE,Feature,Categorical,Race,race,None,no


## 1. Datentypen und Messniveaus
Erster Blick auf unseren Datensatz:

In [19]:
census.info()

<class 'pandas.DataFrame'>
RangeIndex: 199523 entries, 0 to 199522
Data columns (total 42 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   AGE                      199523 non-null  int64  
 1   CLASS_OF_WORKER          199523 non-null  str    
 2   INDUSTRY_CODE            199523 non-null  int64  
 3   OCCUPATION_CODE          199523 non-null  int64  
 4   EDUCATION                199523 non-null  str    
 5   ENROLLED_IN_EDU          199523 non-null  str    
 6   MARITAL_STATUS           199523 non-null  str    
 7   MAJOR_INDUSTRY           199523 non-null  str    
 8   MAJOR_OCCUPATION         199523 non-null  str    
 9   RACE                     199523 non-null  str    
 10  HISPANIC_ORIGIN          199523 non-null  str    
 11  SEX                      199523 non-null  str    
 12  LABOR_UNION_MEMBER       199523 non-null  str    
 13  UNEMPLOYMENT_REASON      199523 non-null  str    
 14  EMPLOYMENT_STAT

### Wichtigste Erkenntnisse zur Datenqualität und Vollständigkeit

1. **Nicht jede Spalte ist vollständig:**
Ein signifikanter Teil des Datensatzes weist Lücken auf. Wir haben hier drei Spalten als Beispiele herausgesucht:
    - **`MIGMTR1`** (Migration Code - Change in MSA) hat nur 99'827 von 199'523 Einträgen (es fehlen knapp 50 % der Daten).
    - **`PEFNTVTY`** (Country of Birth Father) hat 192'810 von 199'523 Einträgen (es fehlen 6.713 Werte).
    - **`GRINST`** (State of Previous Residence) hat 198'815 von 199'523 Einträgen (es fehlen 708 Werte).

2. **Systematische Lücken bei migrationsspezifischen Daten:**
Auffällig ist, dass die Spalten `MIGMTR1`, `MIGMTR3`, `MIGMTR4` und `MIGSUN` alle exakt dieselbe Anzahl an nicht-leeren Werten (99'827) aufweisen. Dies deutet stark darauf hin, dass diese Werte nicht zufällig fehlen sondern aufgrund einer gemeinsamen Abhängigkeit. 

3. **Fehlerhafte Metadaten-Dokumentation (`missing_values` Spalte):**
Ein Problem erkannten wir bei der Dokumentationsspalte `missing_values`. Dort wird bei einigen Spalten fälschlicherweise der Wert `no` angegeben, obwohl `df.info()` eindeutig Lücken belegt. Zum Beispiel die Spalte `PENATVTY`, bei der 3'393 Werte fehlen. Man darf sich bei der Vorverarbeitung also nicht blind auf die textuelle Dokumentation verlassen.

4. **Der `dtype` ist nicht das Messniveau:** Zum Beispiel ist `ADTOCC` als `int64` typisiert, aber es handelt sich um eine nominale Skala (Kategoriencodes für Berufe ohne jegliche Rangfolge). Hier würde Pandas zwar einen Mittelwert berechnen, dies wäre mathematisch aber nicht zulässig, da es keinen „durchschnittlichen Beruf“ gibt.

### Spaltenübersicht

Die folgende Tabelle erstellt einen Überblick über alle 42 Spalten in unserem Datensatz inklusive einiger Beispiele:

In [22]:
def spalten_ueberblick(df: pd.DataFrame, beispiele: int = 3) -> pd.DataFrame:
    """Kompakter Steckbrief je Spalte: dtype, Anzahl eindeutiger Werte, fehlende Werte, Beispiele.

    Erster Schritt bei jedem unbekannten Datensatz.
    """
    zeilen = []
    for spalte in df.columns:
        s = df[spalte]
        werte = s.dropna().unique()[:beispiele]
        zeilen.append(
            {
                "dtype": str(s.dtype),
                "eindeutig": s.nunique(dropna=True),
                "fehlend": int(s.isna().sum()),
                "fehlend_%": round(s.isna().mean() * 100, 1),
                "beispiele": ", ".join(map(str, werte)),
            }
        )
    return pd.DataFrame(zeilen, index=df.columns)


spalten_ueberblick(census)

,dtype,eindeutig,fehlend,fehlend_%,beispiele
AGE,int64,91,0,0.0,"73, 58, 18"
CLASS_OF_WORKER,str,9,0,0.0,"Not in universe, Self-employed-not incorpora..."
INDUSTRY_CODE,int64,52,0,0.0,"0, 4, 40"
OCCUPATION_CODE,int64,47,0,0.0,"0, 34, 10"
EDUCATION,str,17,0,0.0,"High school graduate, Some college but no de..."
ENROLLED_IN_EDU,str,3,0,0.0,"Not in universe, High school, College or un..."
MARITAL_STATUS,str,7,0,0.0,"Widowed, Divorced, Never married"
MAJOR_INDUSTRY,str,24,0,0.0,"Not in universe or children, Construction, ..."
MAJOR_OCCUPATION,str,15,0,0.0,"Not in universe, Precision production craft ..."
RACE,str,5,0,0.0,"White, Asian or Pacific Islander, Amer Indi..."


### Die vier Messniveaus

| Skala | Eigenschaft | Erlaubte Statistik |
|---|---|---|
| **Nominal** | Kategorien ohne Ordnung | Modus, Häufigkeiten |
| **Ordinal** | Rangfolge, Abstände undefiniert | zusätzlich Median, Perzentile |
| **Intervall** | gleiche Abstände, kein natürlicher Nullpunkt | zusätzlich Mittelwert, Standardabweichung |
| **Ratio** | absoluter Nullpunkt | zusätzlich Verhältnisse, geometrisches Mittel |

In [ ]:
# Inhaltliche Zuordnung. 
MESSNIVEAU = {
    "AAGE": "ratio",          # 0 Jahre ist ein echter Nullpunkt
    "ACLSWKR": "nominal",     # Arbeiterklasse (z.B. Private, Federal Gov)
    "ADTINK": "nominal",      # Industrie-Code, int64 aber reiner Code ohne Rang
    "ADTOCC": "nominal",      # Berufs-Code, int64 aber reiner Code ohne Rang
    "AHGA": "ordinal",        # Bildungsabschluss (hat eine logische Rangfolge)
    "AHSCOL": "nominal",      # Einschreibung Schule/Uni
    "AMARITL": "nominal",     # Familienstand
    "AMJIND": "nominal",      # Hauptbranche
    "AMJOCC": "nominal",      # Hauptberufsgruppe
    "ARACE": "nominal",       # Ethnie/Rasse
    "AREORGN": "nominal",     # Hispanische Herkunft
    "ASEX": "nominal",        # Geschlecht
    "AUNMEM": "nominal",      # Gewerkschaftsmitgliedschaft
    "AUNTYPE": "nominal",     # Grund für Arbeitslosigkeit
    "AWKSTAT": "nominal",     # Vollzeit-/Teilzeit-Status
    "CAPGAIN": "ratio",       # Kapitalgewinne (0 $ ist ein echter Nullpunkt)
    "GAPLOSS": "ratio",       # Kapitalverluste (0 $ ist ein echter Nullpunkt)
    "DIVVAL": "ratio",        # Dividenden aus Aktien (0 $ ist ein echter Nullpunkt)
    "FILESTAT": "nominal",    # Steuerstatus
    "GRINREG": "nominal",     # Region des vorherigen Wohnsitzes
    "GRINST": "nominal",      # Bundesstaat des vorherigen Wohnsitzes
    "HHDFMX": "nominal",      # Detaillierte Haushaltsstruktur
    "HHDREL": "nominal",      # Beziehung im Haushalt
    "MARSUPWRT": "ratio",     # Gewichtung / Instance Weight
    "MIGMTR1": "nominal",     # Migrationscode (MSA)
    "MIGMTR3": "nominal",     # Migrationscode (Region)
    "MIGMTR4": "nominal",     # Migrationscode (innerhalb Region)
    "MIGSAME": "nominal",     # Gleiches Haus wie vor 1 Jahr (Ja/Nein-Code)
    "MIGSUN": "nominal",      # Migration (Sunbelt)
    "NOEMP": "ratio",         # Anzahl der Beschäftigten beim Arbeitgeber
    "PARENT": "nominal",      # Elternpräsenz im Haushalt
    "PEFNTVTY": "nominal",    # Geburtsland Vater
    "PEMNTVTY": "nominal",    # Geburtsland Mutter
    "PENATVTY": "nominal",    # Geburtsland Selbst
    "PRCITSHP": "nominal",    # Staatsbürgerschaft
    "SEOTR": "nominal",       # Selbstständigkeit (Code)
    "VETQVA": "nominal",      # Fragebogen für Veteranen
    "VETYN": "nominal",       # Veteranenstatus (Ja/Nein)
    "WKSWORK": "ratio",       # Gearbeitete Wochen im Jahr (0 ist echter Nullpunkt)
    "AHRSPAY": "ratio",       # Stundenlohn (0 $ ist echter Nullpunkt)
    "year": "intervall",      # Jahreszahlen haben keinen natürlichen Nullpunkt
    "income": "ordinal",      # Einkommensklasse (z.B. <50k, >=50k), hat Rangfolge
}

zuordnung = pd.DataFrame(
    {
        "dtype": census.dtypes.astype(str),
        "messniveau": pd.Series(MESSNIVEAU),
    }
)
zuordnung["mittelwert_zulaessig"] = zuordnung["messniveau"].isin(["intervall", "ratio"])
zuordnung["mittelwert"] = [
    census[spalte].mean() if zulaessig else "-"
    for spalte, zulaessig in zuordnung["mittelwert_zulaessig"].items()
]
zuordnung

,dtype,messniveau,mittelwert_zulaessig,mittelwert
AAGE,int64,ratio,True,34.494199
ACLSWKR,str,nominal,False,-
ADTINK,int64,nominal,False,-
ADTOCC,int64,nominal,False,-
AHGA,str,ordinal,False,-
AHSCOL,str,nominal,False,-
AMARITL,str,nominal,False,-
AMJIND,str,nominal,False,-
AMJOCC,str,nominal,False,-
ARACE,str,nominal,False,-


Wie unsere Übersicht zeigt, führt der Datentyp `int64` dazu, dass `pandas` mathematisch völlig unzulässige Mittelwerte für rein nominale Variablen (wie z.B. den Berufscode `ADTOCC`) berechnen würde. Es gibt jedoch auch Spalten wie z.B. `CAPGAIN` (Kapitalgewinne), die den `int64` Datentyp haben, bei denen ein Mittelwert durchaus sinnvoll ist. 

---

## 2. Ursprung der Daten und Bias

### Wie sind unsere Daten entstanden?

Um die vier Fragen zu dem Datenset (aus dem Referenznotebook) zu beantworten haben wir die Dokumentation auf UCI und die Beschreibung der zugrundeliegenden Umfrage des U.S. Census Bureau angeschaut.

**1. Wer ist in den Daten und wer fehlt?**
Die Daten stammen aus dem "Current Population Survey (CPS)" der Jahre 1994 und 1995. Befragt werden Haushalte, die Zielpopulation ist die zivile Bevölkerung der USA, die nicht in Einrichtungen lebt. Also fehlen grundsätzlich alle Personen, die in Gefängnissen, Pflegeheimen oder Kasernen leben. Auch zum Beispiel Obdachlose ohne festen Wohnsitz fehlen. Kinder sind hingegen enthalten, obwohl viele Fragen für sie gar nicht gelten/sinnvoll sind (daher die vielen "Not in universe"-Werte).

**2. Gibt es einen Selektionsprozess?**
Ja, gibt es:
- Die Teilnahme am CPS ist freiwillig. Haushalte, die nicht antworten, fehlen. Ebenfalls werden fehlende Einkommensangaben vom Census Bureau teilweise imputiert.
- Die Stichprobe ist geschichtet, darum gibt es das Gewicht `MARSUPWRT` (siehe nächster Abschnitt).

**3. Wie wurde gemessen?**
Alle Angaben sind Selbstauskünfte aus Interviews. Oft beantwortet eine Person die Fragen für den ganzen Haushalt. Beim Einkommen erwarten wir deshalb eher zu tiefe Werte, weil Einkommen anderer Haushaltsmitglieder oder Kapitalerträge oft nicht genau bekannt sind. Zusätzlich werden sehr hohe Einkommen im öffentlichen CPS aus Datenschutzgründen nach oben begrenzt, was hohe Werte ebenfalls nach unten verzerrt.

**4. Gibt es unmögliche oder verdächtige Werte?**
<br>Das haben wir direkt im Code unten geprüft:

In [ ]:
# Kinder im Datensatz (CPS-Einkommensfragen gelten erst ab 15 Jahren)
print(f"Anteil unter 15 Jahren: {(census['AAGE'] < 15).mean():.1%}")
print(f"Alter min/max: {census['AAGE'].min()} / {census['AAGE'].max()}\n")

# Doppelte Zeilen (ohne Gewicht, da dieses pro Person verschieden sein kann)
print(f"Doppelte Zeilen: {census.drop(columns='MARSUPWRT').duplicated().sum()}\n")

# Fehlen die Migrationswerte in einem bestimmten Jahr?
print("Anteil fehlend MIGMTR1 pro Jahr:")
print(census.groupby("year")["MIGMTR1"].apply(lambda s: s.isna().mean()), "\n")

# Platzhalter "Not in universe" in den Textspalten
text_spalten = census.select_dtypes(exclude="number")
not_in_universe = text_spalten.apply(lambda s: s.str.strip().str.startswith("Not in universe").mean() * 100)
print("Anteil 'Not in universe' in %:")
print(not_in_universe[not_in_universe > 0].sort_values(ascending=False).round(1), "\n")

# Nullen und Maxima bei den Geldvariablen
geld = ["AHRSPAY", "CAPGAIN", "GAPLOSS", "DIVVAL"]
print(pd.DataFrame({
    "anteil_null_%": ((census[geld] == 0).mean() * 100).round(1),
    "max": census[geld].max(),
}))

Anteil unter 15 Jahren: 23.8%
Alter min/max: 0 / 90

Doppelte Zeilen: 46627

Anteil fehlend MIGMTR1 pro Jahr:
year
94    0.0
95    1.0
Name: MIGMTR1, dtype: float64 

Anteil 'Not in universe' in %:
VETQVA     99.0
AUNTYPE    97.0
AHSCOL     93.7
GRINST     92.1
GRINREG    92.1
AUNMEM     90.4
PARENT     72.3
MIGSAME    50.7
AMJIND     50.5
AMJOCC     50.5
ACLSWKR    50.2
MIGSUN     42.1
MIGMTR4     0.8
MIGMTR1     0.8
MIGMTR3     0.8
dtype: float64 

         anteil_null_%    max
AHRSPAY           94.3   9999
CAPGAIN           96.3  99999
GAPLOSS           98.0   4608
DIVVAL            89.4  99999


**Erkenntnisse aus der Werte-Prüfung:**

- **Kinder machen einen grossen Teil aus.** 23.8 % der Personen sind jünger als 15 Jahre, das jüngste Alter ist 0. Für diese Personen sind Fragen zu Arbeit und Einkommen gar nicht gestellt worden. Das Höchstalter liegt bei genau 90, was darauf hindeutet, dass auch das Alter nach oben eventuell begrenzt wurde.

- **Viele Duplikate.** 46'627 Zeilen sind (ohne Gewicht) identisch mit einer anderen Zeile. Aber da wir sehr viele grobe Kategorien haben muss das nicht zwingend ein Fehler sein, da zwei verschiedene Personen schon durchaus dieselben Merkmale haben können.

- **Die Migrationsdaten fehlen für 1995 komplett.** `MIGMTR1` fehlt 1994 nie und 1995 immer. Also fehlen die Werte nicht zufällig, sondern wurden 1995 schlicht nicht erhoben oder nicht mitgeliefert. Aussagen zur Migration gelten für uns also nur für 1994.

- **"Not in universe" ist oft der Normalfall.** Bei `VETQVA`, `AUNTYPE` und `AHSCOL` betrifft der Platzhalter über 90 % der vorhandenen Werte, bei Beruf und Branche (`AMJOCC`, `AMJIND`, `ACLSWKR`) rund die Hälfte. Diese Werte sind keine echte Kategorie, sondern bedeuten, dass die Frage nicht gestellt wurde. Dies reduziert die vorhandene Datenmenge über die entsprechenden Kategorien stark. 

- **Verdächtige/fehldende Geldvariablen.** Zwischen 89 % und 98 % der Werte sind 0, wahrscheinlich meistens weil die Person gar kein solches Einkommen hat. Uns sind auch die Maxima aufgefallen: 99'999 bei `CAPGAIN` und `DIVVAL` sowie 9'999 bei `AHRSPAY` sind sehr wahrscheinlich Obergrenzen und somit vermutlich keine echten Beträge.

**Fazit:** Ein grosser Teil unseres Datensatzes besteht aus Werten, die zwar nicht als fehlend markiert sind, inhaltlich aber "nicht zutreffend" bedeuten. Ungefilterte Mittelwerte würden dadurch stark nach unten gezogen. Für jede weitere Kennzahl legen wir deshalb fest, auf welche Teilpopulation sie sich bezieht, z.B. nur Erwerbstätige oder nur Personen ab 15 Jahren.

### Erkenntnis: Der Datensatz ist gewichtet

Die Daten stammen nicht aus einer einfachen Zufallsstichprobe. Der Census Bureau hat geschichtet gezogen (stratified sampling) und manche Gruppen absichtlich häufiger befragt, als es ihrem Anteil in der Bevölkerung entspricht, damit auch über kleine Gruppen genug Daten vorhanden sind. Diese Gruppen sind in unserem Datensatz deshalb übervertreten.

Ausgeglichen wird das durch die Spalte `MARSUPWRT`. Sie gibt an, wie viele Personen in der Bevölkerung eine Zeile repräsentiert. Eine Zeile mit Gewicht 2000 steht für 2000 Personen, eine mit Gewicht 500 nur für 500. In einem gewichteten Mittelwert zählt die erste Person also viermal so stark.

Also bedeutet das für unsere Analyse:
- Ungewichtete Kennzahlen beschreiben nur die Stichprobe. Für die EDA konnten wir sie zwar verwenden, haben sie aber entsprechend gekennzeichnet.
- Für Aussagen über die Population gewichten wir die Werte mit `MARSUPWRT`.

Nachfolgend haben wir ungewichtete und mit den Census-Gewichten berechnete Mittelwerte verglichen und untersucht, wie sich die Gewichtung auf Kennzahlen und Einkommensanteile auswirkt:

In [ ]:
spalten = ["AAGE", "WKSWORK", "AHRSPAY", "CAPGAIN", "DIVVAL"]
gewicht = census["MARSUPWRT"]

vergleich = pd.DataFrame(
    {
        "ungewichtet": census[spalten].mean(),
        "gewichtet": [np.average(census[s], weights=gewicht) for s in spalten],
    }
)
vergleich["abweichung_%"] = (vergleich["gewichtet"] / vergleich["ungewichtet"] - 1) * 100

# Auch Anteile verschieben sich: z.B. Anteil Personen mit Einkommen über 50k
ueber_50k = census["income"].str.strip() == "50000+."
print(f"Anteil >50k ungewichtet: {ueber_50k.mean():.2%}")
print(f"Anteil >50k gewichtet:   {np.average(ueber_50k, weights=gewicht):.2%}\n")

vergleich.round(2)

Anteil >50k ungewichtet: 6.21%
Anteil >50k gewichtet:   6.41%



,ungewichtet,gewichtet,abweichung_%
AAGE,34.49,34.47,-0.06
WKSWORK,23.17,23.58,1.76
AHRSPAY,55.43,57.37,3.50
CAPGAIN,434.72,441.56,1.57
DIVVAL,197.53,197.52,-0.01


Der Vergleich zeigt, dass die Unterschiede zwischen gewichteten und ungewichteten Werten bei unseren Variablen klein sind. Beim Alter (`AAGE`) und bei den Dividenden (`DIVVAL`) ist die Abweichung praktisch 0%. Am grössten ist sie beim Stundenlohn (`AHRSPAY`) mit 3.5%. Auch der Anteil der Personen mit einem Einkommen über 50k steigt nur leicht von 6.21% auf 6.41%.

Die gewichteten Werte liegen dort, wo es eine Abweichung gibt, jeweils etwas höher. Das deutet darauf hin, dass Gruppen mit tieferem Lohn und weniger Arbeitswochen in der Stichprobe leicht übervertreten sind. Für diese Kennzahlen auf Ebene des gesamten Datensatzes verfälscht die Schichtung das Bild also fast nicht. Trotzdem verwenden wir für Aussagen über die Population die gewichteten Werte, da sich die Abweichungen bei Auswertungen einzelner Untergruppen verstärken könnten.

### Sampling Bias

Um darzustellen, dass eine Stichprobe die Zielpopulation nicht (zwingend) akkurat abbilet haben wir hier dargestellt, was passiert, wenn man nur eine bestimmte Gruppe der Bevölkerung befragt. Hier ein Beispiel für das Alter von Personen mit einem bestimmten Familienstand.

In [ ]:
wahrer_mittelwert = np.average(census["AAGE"], weights=census["MARSUPWRT"])

nach_familienstand = census.groupby("AMARITL")["AAGE"].agg(["size", "mean"])
nach_familienstand["abweichung"] = nach_familienstand["mean"] - wahrer_mittelwert
nach_familienstand = nach_familienstand.sort_values("abweichung")

print(f"Mittleres Alter, alle Personen: {wahrer_mittelwert:.2f}\n")
print(nach_familienstand)

Mittleres Alter, alle Personen: 34.47

                                  size       mean  abweichung
AMARITL                                                      
Never married                    86485  15.922622  -18.551058
Married-A F spouse present         665  31.673684   -2.799996
Separated                         3460  41.760116    7.286435
Married-spouse absent             1518  45.525692   11.052011
Divorced                         12710  46.129583   11.655903
Married-civilian spouse present  84222  46.705433   12.231753
Widowed                          10463  71.750454   37.276774


Hier ist ersichtlich, dass je nachdem, welche Gruppe man befragt, das Durchschnittsalter stark vom wahren Wert abweicht. Bei "Never married" ist der Mittelwert deutlich tiefer, weil in dieser Gruppe auch alle Kinder enthalten sind. Bei "Widowed" liegt er klar darüber, weil Verwitwete meist älter sind. Wenn man also z.B. eine Umfrage nur über einen Kanal macht, der vor allem eine bestimmte Gruppe erreicht, zieht man danach falsche Schlüsse über die ganze Population.